# Bootstrap: Kaggle auth onto this Colab VM
Run the one cell below with the Colab kernel (extension). It reads the stored Colab secret and writes Kaggle credentials **on the VM only**. It prints names, lengths and the VM hostname, never secret values. After it succeeds, the agent drives this runtime headlessly via the Colab MCP API.

In [1]:
import json, os, socket, stat
from pathlib import Path
from google.colab import userdata

def get(name):
    try:
        return userdata.get(name)
    except Exception as e:
        print(f'secret {name}: {type(e).__name__}')
        return None

kd = Path.home() / '.kaggle'; kd.mkdir(exist_ok=True)
# names as stored in this account's Colab Secrets panel (KAGGLE_USERNAME, KAGGLE_KEY, kagKaggle_token)
user, key, tok = get('KAGGLE_USERNAME'), get('KAGGLE_KEY'), get('kagKaggle_token')
wrote = []
if user and key and not key.startswith('KGAT_'):  # classic API key
    (kd / 'kaggle.json').write_text(json.dumps({'username': user, 'key': key})); wrote.append('kaggle.json')
# new-style KGAT_ access token: from the token secret, or KAGGLE_KEY if that holds one
kgat = next((v for v in (tok, key) if v and v.strip().startswith('KGAT_')), None) or tok
if kgat:
    (kd / 'access_token').write_text(kgat.strip()); wrote.append('access_token')
print('formats:', {n: ('KGAT' if v.strip().startswith('KGAT_') else 'classic/other') for n, v in [('key', key), ('token', tok)] if v})
for f in kd.iterdir():
    f.chmod(stat.S_IRUSR | stat.S_IWUSR)
print('wrote:', wrote or 'NOTHING (no secret readable)')
print('lengths:', {n: len(v) for n, v in [('user', user), ('key', key), ('token', tok)] if v})
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle>=2.2'], check=False)
r = subprocess.run(['kaggle', 'datasets', 'files', 'comfyops/gemma129-taskdefs'], capture_output=True, text=True)
print('kaggle auth check rc', r.returncode, '|', (r.stdout or r.stderr).strip().splitlines()[-1:] )
print('hostname:', socket.gethostname())
Path('/content/_bootstrap_ok').write_text(socket.gethostname())  # marker the agent checks via the API


secret KAGGLE_USERNAME: TimeoutException
secret KAGGLE_KEY: TimeoutException
secret KAGGLE_API_TOKEN: TimeoutException
secret KAGGLE_JSON: TimeoutException
wrote: NOTHING (no secret readable)
lengths: {}
hostname: 9244d4eba92d


12